# Llama-2-7B QLoRA fine-tune → Hugging Face

Kaggle **T4 x2**, Internet **On**. Run cell 1, **Restart session**, then run the rest top-to-bottom.

Set an HF write token as the Kaggle secret `HF_TOKEN` (Add-ons → Secrets), or paste it into cell 3.

Output: <https://huggingface.co/abdur-rahman77/llama2-7b-qlora-guanaco>

In [ ]:
# CELL 1 - pinned stack. transformers 5.x breaks the bitsandbytes 4-bit load path (Aug 2026);
# 4.56 is the last known-good. After this finishes: Run menu -> Restart session.
!pip install -q "transformers==4.56.2" "trl==0.21.0" "peft==0.15.2" \
    "accelerate==1.10.1" "bitsandbytes==0.47.0" "datasets>=3.0"
print("installed - RESTART SESSION now, then run cell 2")

In [ ]:
# CELL 2 - after restart: confirm versions + pick a free GPU
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, transformers, trl, peft, accelerate, bitsandbytes
print("transformers", transformers.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| accelerate", accelerate.__version__,
      "| bnb", bitsandbytes.__version__)
GPU_ID, best = 0, -1
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} free {free/1024**3:.2f} / {total/1024**3:.2f} GiB")
    if free > best: GPU_ID, best = i, free
print("=> using cuda:", GPU_ID)
assert best/1024**3 > 8, "no GPU has >8 GiB free -> Restart session and rerun"

In [ ]:
# CELL 3 - auth + config
torch.cuda.set_device(GPU_ID)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = ""          # <-- or paste token here
assert HF_TOKEN, "set Kaggle secret HF_TOKEN or paste it above"
os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
from huggingface_hub import login, whoami
login(token=HF_TOKEN)
print("HF auth ok:", whoami()["name"])

MODEL_NAME   = "NousResearch/Llama-2-7b-chat-hf"
DATASET_NAME = "mlabonne/guanaco-llama2-1k"
MODEL_REPO   = "abdur-rahman77/llama2-7b-qlora-guanaco"
ADAPTER_DIR  = "/kaggle/working/llama2-7b-qlora-adapter"
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 16, 0.05
EPOCHS, BS, GRAD_ACCUM, LR, MAX_SEQ_LEN = 1, 1, 8, 2e-4, 256

In [ ]:
# CELL 4 - dataset
from datasets import load_dataset
dataset = load_dataset(DATASET_NAME, split="train")
print(dataset, "\n", dataset[0]["text"][:200])

In [ ]:
# CELL 5 - 4-bit base model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": GPU_ID},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
free, total = torch.cuda.mem_get_info(GPU_ID)
print(f"loaded on cuda:{GPU_ID}  free {free/1024**3:.2f} / {total/1024**3:.2f} GiB")

In [ ]:
# CELL 6 - LoRA config
peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM",
)

In [ ]:
# CELL 7 - train (~80 min on one T4)
from trl import SFTTrainer, SFTConfig
import inspect

cfg = dict(
    output_dir="/kaggle/working/out",
    num_train_epochs=EPOCHS, per_device_train_batch_size=BS,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
    lr_scheduler_type="cosine", warmup_ratio=0.03, logging_steps=10,
    save_strategy="no", bf16=True, fp16=False, optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="text", packing=False, report_to="none",
)
sig = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in sig:       cfg["max_length"] = MAX_SEQ_LEN
elif "max_seq_length" in sig:  cfg["max_seq_length"] = MAX_SEQ_LEN
sft_config = SFTConfig(**cfg)

try:
    trainer = SFTTrainer(model=model, train_dataset=dataset, peft_config=peft_config,
                         args=sft_config, processing_class=tokenizer)
except TypeError:
    trainer = SFTTrainer(model=model, train_dataset=dataset, peft_config=peft_config,
                         args=sft_config, tokenizer=tokenizer)

trainer.args._n_gpu = 1   # T4 x2 -> don't wrap in DataParallel
trainer.train()

In [ ]:
# CELL 8 - save adapter
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(sorted(os.listdir(ADAPTER_DIR)))

In [ ]:
# CELL 9 - model card + push
card = (
    "---\n"
    f"base_model: {MODEL_NAME}\n"
    "library_name: peft\nlicense: llama2\n"
    f"datasets:\n  - {DATASET_NAME}\n"
    "pipeline_tag: text-generation\n"
    "tags: [qlora, lora, peft, llama-2, instruction-tuning]\n"
    "---\n\n"
    "# Llama-2-7B Chat - QLoRA adapter (Guanaco-1k)\n\n"
    f"LoRA adapter for `{MODEL_NAME}`, QLoRA (4-bit NF4, double-quant, bf16) on "
    f"`{DATASET_NAME}` (1,000 multilingual instructions). Final loss 1.47 (from 2.06), 1 epoch / 125 steps.\n\n"
    "## Training\n\n"
    f"- LoRA r/alpha/dropout {LORA_R}/{LORA_ALPHA}/{LORA_DROPOUT}, targets q_proj,v_proj\n"
    f"- eff. batch {BS*GRAD_ACCUM}, lr {LR} cosine, max_seq {MAX_SEQ_LEN}, paged_adamw_32bit, Kaggle T4\n"
    "- transformers 4.56, trl 0.21, peft 0.15, bitsandbytes 0.47\n\n"
    "Load with `peft.PeftModel.from_pretrained` on top of the base model.\n"
)
open(f"{ADAPTER_DIR}/README.md", "w").write(card)

from huggingface_hub import create_repo, upload_folder
create_repo(MODEL_REPO, repo_type="model", exist_ok=True, private=False)
upload_folder(folder_path=ADAPTER_DIR, repo_id=MODEL_REPO, repo_type="model",
              commit_message="QLoRA adapter (Guanaco-1k), loss 1.47")
print("pushed -> https://huggingface.co/" + MODEL_REPO)